# PIRO - Notebook de Treino das CNNs

**Projeto:** Plataforma Integrada de Resposta Orbital (PIRO)
**Disciplina:** Applied Computer Vision (ACV) - FIAP GS 2026
**Objetivo:** Treinar e comparar duas redes neurais convolucionais **do zero** para classificar imagens como `fire` ou `nofire`.

## Antes de rodar

1. **Ativar GPU**: Settings -> Accelerator -> GPU T4 x2 ou P100
2. **Anexar o dataset**: + Add Input -> `brsdincer/wildfire-detection-image-data`
3. **Run All** na ordem das células

## Estrutura do notebook

| Seção | O que faz |
|-------|-----------|
| 1 | Setup, imports, reprodutibilidade |
| 2 | Carregamento e divisão do dataset (70/15/15 estratificado) |
| 3 | Class weights e generators com augmentation |
| 4 | CNN1 - baseline (Flatten + Dense grande) |
| 5 | CNN2 - profunda com BatchNorm + GlobalAveragePooling |
| 6 | Avaliação final, matrizes de confusão, comparação |
| 7 | Análise qualitativa de erros (FN, FP, TP, TN) |
| 8 | Salvar pesos e gráficos para o repositório |


## 1. Setup e reprodutibilidade

In [ ]:
import os
import random
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs disponiveis: {len(gpus)}")
for gpu in gpus:
    print(f"  - {gpu.name}")

if not gpus:
    print("\nAVISO: nenhuma GPU detectada. Treino vai ser MUITO lento.")
    print("Habilite GPU em Settings -> Accelerator no Kaggle.")


### Configurações globais

In [ ]:
DATA_DIR = Path('/kaggle/input/datasets/brsdincer/wildfire-detection-image-data/forest_fire/Training and Validation')
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(exist_ok=True)

IMG_SIZE = (128, 128)
BATCH_SIZE = 32
CHANNELS = 3
EXTENSOES_VALIDAS = {'.jpg', '.jpeg', '.png', '.webp'}

# Ordem das classes - forcamos nofire=0, fire=1 para que
# probabilidade alta = "tem fogo" (mais intuitivo)
ORDEM_CLASSES = ['nofire', 'fire']
NOMES_LEGIVEIS = {'nofire': 'Sem Fogo', 'fire': 'Fogo'}

print(f"Dataset em: {DATA_DIR}")
print(f"Tamanho de imagem: {IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Mapeamento de classes: 0 = {ORDEM_CLASSES[0]}, 1 = {ORDEM_CLASSES[1]}")


## 2. Carregamento e divisão do dataset

Constrói um DataFrame com `caminho` e `classe` de cada imagem, depois divide
em treino/validação/teste com **estratificação**. Usamos `random_state=42`
para reprodutibilidade.

In [ ]:
todos_arquivos = []
for classe_dir in sorted(DATA_DIR.iterdir()):
    if not classe_dir.is_dir():
        continue
    for img_path in classe_dir.iterdir():
        if img_path.suffix.lower() in EXTENSOES_VALIDAS:
            todos_arquivos.append({
                'caminho': str(img_path),
                'classe': classe_dir.name
            })

df_total = pd.DataFrame(todos_arquivos)
print(f"Total de imagens: {len(df_total)}")
print(f"\nDistribuicao de classes:")
print(df_total['classe'].value_counts())
print(f"\nProporcao: {(df_total['classe'].value_counts(normalize=True) * 100).round(1).to_dict()}")


In [ ]:
# Split 70/15/15 estratificado
df_treino, df_temp = train_test_split(
    df_total,
    test_size=0.30,
    stratify=df_total['classe'],
    random_state=SEED,
)
df_val, df_teste = train_test_split(
    df_temp,
    test_size=0.50,
    stratify=df_temp['classe'],
    random_state=SEED,
)

df_treino = df_treino.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_teste = df_teste.reset_index(drop=True)

print(f"Treino: {len(df_treino)} imagens")
print(f"  {df_treino['classe'].value_counts().to_dict()}")
print(f"\nValidacao: {len(df_val)} imagens")
print(f"  {df_val['classe'].value_counts().to_dict()}")
print(f"\nTeste: {len(df_teste)} imagens")
print(f"  {df_teste['classe'].value_counts().to_dict()}")


## 3. Class weights e generators

O dataset brsdincer é **praticamente balanceado** (50.7% fire / 49.3% nofire),
então class weights ficam próximos de 1.0. Mantemos o cálculo por boa prática
e generalidade do código.

A augmentation de treino é **moderada** (rotações leves, flip horizontal,
brightness controlado). Augmentation agressiva foi testada e descartada
após observação de underfitting em modelos regularizados.

In [ ]:
# Class weights balanceados
pesos = compute_class_weight(
    'balanced',
    classes=np.array(ORDEM_CLASSES),
    y=df_treino['classe'].values
)
class_weight = {i: float(p) for i, p in enumerate(pesos)}

print("Class weights computados:")
for idx, classe in enumerate(ORDEM_CLASSES):
    print(f"  {idx} ({NOMES_LEGIVEIS[classe]}): peso = {class_weight[idx]:.3f}")


In [ ]:
# Augmentation moderada - apenas para treino
train_aug = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.08,
    height_shift_range=0.08,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.85, 1.15],
    fill_mode='nearest',
)

# Para val/teste so normalizacao
plain_rescale = ImageDataGenerator(rescale=1./255)

train_gen = train_aug.flow_from_dataframe(
    df_treino,
    x_col='caminho', y_col='classe',
    classes=ORDEM_CLASSES,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=SEED,
)

val_gen = plain_rescale.flow_from_dataframe(
    df_val,
    x_col='caminho', y_col='classe',
    classes=ORDEM_CLASSES,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
)

test_gen = plain_rescale.flow_from_dataframe(
    df_teste,
    x_col='caminho', y_col='classe',
    classes=ORDEM_CLASSES,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
)

print(f"\nMapeamento de classes no generator: {train_gen.class_indices}")


### Visualizar uma amostra após augmentation

In [ ]:
imgs, labels = next(train_gen)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for i, ax in enumerate(axes.flatten()):
    if i < len(imgs):
        ax.imshow(imgs[i])
        rotulo = NOMES_LEGIVEIS[ORDEM_CLASSES[int(labels[i])]]
        ax.set_title(f'{rotulo} (y={int(labels[i])})', fontsize=10)
    ax.axis('off')

fig.suptitle('Amostras do treino apos augmentation', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'amostras_augmentation.png', dpi=100, bbox_inches='tight')
plt.show()

train_gen.reset()  # resetar estado antes do treino


## 4. CNN1 - Baseline

Arquitetura simples: dois blocos Conv+Pool, depois `Flatten -> Dense`.
A maior parte dos parâmetros (~99%) fica na camada Dense logo após o Flatten.
Serve como baseline para comparação com a CNN2 profunda.

In [ ]:
def build_cnn1(input_shape=(*IMG_SIZE, CHANNELS)):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Bloco 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        # Bloco 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        # Classificador
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid'),
    ], name='cnn1_baseline')

    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', Precision(name='precision'), Recall(name='recall')]
    )
    return model

cnn1 = build_cnn1()
cnn1.summary()


### Treinar CNN1

Sem callbacks. 20 epocas para ver o comportamento natural da rede.

In [ ]:
historia_cnn1 = cnn1.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    class_weight=class_weight,
    verbose=1,
)

cnn1.save(OUTPUT_DIR / 'cnn1_baseline.keras')
with open(OUTPUT_DIR / 'historia_cnn1.json', 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in historia_cnn1.history.items()}, f)


### Curvas de treino - CNN1

In [ ]:
def plot_curvas_treino(historia, titulo, ax_acc, ax_loss):
    h = historia.history
    epocas = range(1, len(h['accuracy']) + 1)

    ax_acc.plot(epocas, h['accuracy'], 'o-', label='treino')
    ax_acc.plot(epocas, h['val_accuracy'], 's-', label='validacao')
    ax_acc.set_title(f'{titulo} - Acuracia')
    ax_acc.set_xlabel('Epoca')
    ax_acc.set_ylabel('Acuracia')
    ax_acc.legend()
    ax_acc.set_ylim([0, 1])

    ax_loss.plot(epocas, h['loss'], 'o-', label='treino')
    ax_loss.plot(epocas, h['val_loss'], 's-', label='validacao')
    ax_loss.set_title(f'{titulo} - Loss')
    ax_loss.set_xlabel('Epoca')
    ax_loss.set_ylabel('Loss')
    ax_loss.legend()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_curvas_treino(historia_cnn1, 'CNN1', axes[0], axes[1])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'curvas_cnn1.png', dpi=100, bbox_inches='tight')
plt.show()


## 5. CNN2 - Profunda e regularizada

Arquitetura inspirada em VGG com 6 diferenças intencionais em relação à CNN1:

1. **3 blocos convolucionais** (vs 2) - hierarquia maior de features
2. **Duas Conv2D por bloco** (estilo VGG) - mais não-linearidade entre poolings
3. **BatchNormalization** - estabiliza treino, permite lr maior, leve regularização
4. **GlobalAveragePooling2D** no lugar de Flatten - elimina ~8M parâmetros
5. **Dropout estratificado** (0.25 -> 0.25 -> 0.3 -> 0.5)
6. **Callbacks** - EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

In [ ]:
def build_cnn2(input_shape=(*IMG_SIZE, CHANNELS)):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Bloco 1
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Bloco 2
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Bloco 3
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        # Classificador enxuto
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid'),
    ], name='cnn2_profunda')

    model.compile(
        optimizer=Adam(learning_rate=5e-4),
        loss='binary_crossentropy',
        metrics=['accuracy', Precision(name='precision'), Recall(name='recall')]
    )
    return model

cnn2 = build_cnn2()
cnn2.summary()


### Treinar CNN2 com callbacks

- **EarlyStopping**: interrompe se val_loss não melhorar por 15 épocas
- **ReduceLROnPlateau**: divide lr por 2 se val_loss travar por 5 épocas
- **ModelCheckpoint**: salva o melhor modelo automaticamente

LR inicial menor (5e-4) e patience maior estabilizam o treino com BatchNorm
em batches relativamente pequenos.

In [ ]:
train_gen.reset()
val_gen.reset()

callbacks_cnn2 = [
    EarlyStopping(monitor='val_loss', patience=15,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=5, min_lr=1e-6, verbose=1),
    ModelCheckpoint(str(OUTPUT_DIR / 'cnn2_melhor.keras'),
                    monitor='val_accuracy',
                    save_best_only=True, verbose=0),
]

historia_cnn2 = cnn2.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    class_weight=class_weight,
    callbacks=callbacks_cnn2,
    verbose=1,
)

cnn2.save(OUTPUT_DIR / 'cnn2_final.keras')
with open(OUTPUT_DIR / 'historia_cnn2.json', 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in historia_cnn2.history.items()}, f)


### Curvas de treino - CNN2

Note o fenômeno de **BatchNorm warmup**: val_loss alta nas primeiras
épocas (estatísticas móveis não convergiram), seguida de queda brusca
quando o BN se estabiliza (~época 10-12). Comportamento esperado.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_curvas_treino(historia_cnn2, 'CNN2', axes[0], axes[1])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'curvas_cnn2.png', dpi=100, bbox_inches='tight')
plt.show()


## 6. Avaliação no conjunto de teste

Função reutilizável que retorna **acurácia, precisão, recall, F1**
e a **matriz de confusão**, além de gerar previsões para análise de erros.

In [ ]:
def avaliar_modelo(model, gen, nome):
    gen.reset()
    y_prob = model.predict(gen, verbose=0).ravel()
    y_pred = (y_prob > 0.5).astype(int)
    y_true = np.array(gen.classes).astype(int)  # garante array numpy int

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    print(f"=== {nome} ===")
    print(f"Acuracia:  {acc:.4f}")
    print(f"Precisao:  {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"\nClassification report:")
    print(classification_report(y_true, y_pred,
                                 target_names=[NOMES_LEGIVEIS[c] for c in ORDEM_CLASSES]))

    return {
        'nome': nome,
        'acuracia': acc,
        'precisao': prec,
        'recall': rec,
        'f1': f1,
        'matriz_confusao': cm,
        'y_true': y_true,
        'y_pred': y_pred,
        'y_prob': y_prob,
    }


In [ ]:
resultado_cnn1 = avaliar_modelo(cnn1, test_gen, 'CNN1 - Baseline')


In [ ]:
# Carregar o MELHOR checkpoint da CNN2 (best val_accuracy)
cnn2_melhor = tf.keras.models.load_model(OUTPUT_DIR / 'cnn2_melhor.keras')
resultado_cnn2 = avaliar_modelo(cnn2_melhor, test_gen, 'CNN2 - Profunda')


### Tabela comparativa

In [ ]:
df_comparacao = pd.DataFrame([
    {
        'Modelo': resultado_cnn1['nome'],
        'Acuracia': resultado_cnn1['acuracia'],
        'Precisao': resultado_cnn1['precisao'],
        'Recall': resultado_cnn1['recall'],
        'F1-score': resultado_cnn1['f1'],
        'Parametros': cnn1.count_params(),
    },
    {
        'Modelo': resultado_cnn2['nome'],
        'Acuracia': resultado_cnn2['acuracia'],
        'Precisao': resultado_cnn2['precisao'],
        'Recall': resultado_cnn2['recall'],
        'F1-score': resultado_cnn2['f1'],
        'Parametros': cnn2.count_params(),
    },
])

df_comparacao.to_csv(OUTPUT_DIR / 'comparacao_modelos.csv', index=False)
df_comparacao.style.format({
    'Acuracia': '{:.4f}',
    'Precisao': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-score': '{:.4f}',
    'Parametros': '{:,}',
}).background_gradient(subset=['Acuracia', 'F1-score'], cmap='Greens')


### Matrizes de confusão lado a lado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, resultado in zip(axes, [resultado_cnn1, resultado_cnn2]):
    nomes = [NOMES_LEGIVEIS[c] for c in ORDEM_CLASSES]
    sns.heatmap(resultado['matriz_confusao'], annot=True, fmt='d',
                cmap='Blues', cbar=False, ax=ax,
                xticklabels=nomes, yticklabels=nomes)
    ax.set_title(f"{resultado['nome']}\nAcuracia: {resultado['acuracia']:.4f} | Recall: {resultado['recall']:.4f}")
    ax.set_xlabel('Previsto')
    ax.set_ylabel('Verdadeiro')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'matrizes_confusao.png', dpi=100, bbox_inches='tight')
plt.show()


### Curvas de treino - comparação lado a lado

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
plot_curvas_treino(historia_cnn1, 'CNN1', axes[0][0], axes[0][1])
plot_curvas_treino(historia_cnn2, 'CNN2', axes[1][0], axes[1][1])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'curvas_comparacao.png', dpi=100, bbox_inches='tight')
plt.show()


## 7. Análise qualitativa de erros

Tipos de erro mais importantes para o PIRO:
- **Falsos negativos** (fogo real classificado como sem fogo) — pior caso, alerta perdido
- **Falsos positivos** (sem fogo classificado como fogo) — menos grave, mas gera alarme falso

In [ ]:
def mostrar_exemplos(resultado, gen, tipo, n=6):
    '''Mostra n exemplos de TP, TN, FP ou FN.'''
    y_true = np.array(resultado['y_true']).astype(int)
    y_pred = np.array(resultado['y_pred']).astype(int)
    y_prob = np.array(resultado['y_prob']).astype(float)
    caminhos = gen.filepaths

    mapa_tipo = {
        'TP': ((y_true == 1) & (y_pred == 1), 'TP - fogo detectado corretamente'),
        'TN': ((y_true == 0) & (y_pred == 0), 'TN - sem fogo, classificado corretamente'),
        'FP': ((y_true == 0) & (y_pred == 1), 'FP - alarme falso'),
        'FN': ((y_true == 1) & (y_pred == 0), 'FN - fogo NAO detectado (critico!)'),
    }
    mascara, titulo = mapa_tipo[tipo]
    idx = np.where(mascara)[0]
    print(f"Encontrados: {len(idx)} exemplos de {tipo}")

    if len(idx) == 0:
        return

    n_mostrar = min(n, len(idx))
    amostra = np.random.RandomState(SEED).choice(idx, n_mostrar, replace=False)

    cols = min(n_mostrar, 3)
    rows = (n_mostrar + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows), squeeze=False)
    axes_flat = axes.flatten()

    for ax_i, i in enumerate(amostra):
        img = Image.open(caminhos[i])
        axes_flat[ax_i].imshow(img)
        axes_flat[ax_i].set_title(f'prob fogo = {y_prob[i]:.3f}', fontsize=10)
        axes_flat[ax_i].axis('off')

    for ax_i in range(len(amostra), len(axes_flat)):
        axes_flat[ax_i].axis('off')

    fig.suptitle(titulo, fontsize=13)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'erros_{tipo}.png', dpi=100, bbox_inches='tight')
    plt.show()


In [ ]:
# Resumo da matriz de confusao da CNN2
y_true = resultado_cnn2['y_true']
y_pred = resultado_cnn2['y_pred']
print("CNN2 - Quantitativos:")
print(f"  TP: {((y_true == 1) & (y_pred == 1)).sum()}")
print(f"  TN: {((y_true == 0) & (y_pred == 0)).sum()}")
print(f"  FP: {((y_true == 0) & (y_pred == 1)).sum()}")
print(f"  FN: {((y_true == 1) & (y_pred == 0)).sum()}")


In [ ]:
# Falsos negativos - erro mais critico
mostrar_exemplos(resultado_cnn2, test_gen, 'FN', n=6)


In [ ]:
# Falsos positivos
mostrar_exemplos(resultado_cnn2, test_gen, 'FP', n=6)


In [ ]:
# Acertos
mostrar_exemplos(resultado_cnn2, test_gen, 'TP', n=4)
mostrar_exemplos(resultado_cnn2, test_gen, 'TN', n=4)


## 8. Arquivos gerados e salvamento do modelo final

In [ ]:
# Salvar o modelo escolhido como 'modelo_final_piro.keras' para uso no Streamlit
import shutil
shutil.copy(OUTPUT_DIR / 'cnn2_melhor.keras', OUTPUT_DIR / 'modelo_final_piro.keras')
print("Modelo final salvo como modelo_final_piro.keras")


In [ ]:
arquivos = sorted(OUTPUT_DIR.glob('*'))
print(f"Arquivos gerados em {OUTPUT_DIR}:\n")
for f in arquivos:
    if f.is_file():
        tamanho_kb = f.stat().st_size / 1024
        if tamanho_kb > 1024:
            print(f"  {f.name:<35} {tamanho_kb/1024:>7.1f} MB")
        else:
            print(f"  {f.name:<35} {tamanho_kb:>7.1f} KB")


In [ ]:
ganho_acc = (resultado_cnn2['acuracia'] - resultado_cnn1['acuracia']) * 100
reducao_params = (1 - cnn2.count_params() / cnn1.count_params()) * 100

print(f'''Conclusao tecnica:

CNN1: acuracia {resultado_cnn1["acuracia"]:.4f} | F1 {resultado_cnn1["f1"]:.4f} | {cnn1.count_params():,} params
CNN2: acuracia {resultado_cnn2["acuracia"]:.4f} | F1 {resultado_cnn2["f1"]:.4f} | {cnn2.count_params():,} params

Diferenca de acuracia: {ganho_acc:+.2f} pontos percentuais
Reducao de parametros: {reducao_params:.1f}% ({cnn1.count_params():,} -> {cnn2.count_params():,})

A CNN2 e o modelo escolhido pelos seguintes criterios:
1. Recall (metrica de negocio): {resultado_cnn2["recall"]:.4f} vs {resultado_cnn1["recall"]:.4f}
2. F1-score: {resultado_cnn2["f1"]:.4f} vs {resultado_cnn1["f1"]:.4f}
3. Viabilidade de deploy embarcado ({reducao_params:.1f}% menos parametros)
''')


## Próximos passos

Após rodar este notebook até o fim, você terá em `/kaggle/working/`:

- `modelo_final_piro.keras` - peso final do modelo escolhido (vai pro repo + Streamlit)
- `cnn1_baseline.keras` - peso da CNN1 (comparação)
- `cnn2_final.keras` e `cnn2_melhor.keras` - CNN2 (versão final e melhor checkpoint)
- `comparacao_modelos.csv` - tabela para o relatório
- `matrizes_confusao.png`, `curvas_comparacao.png`, `amostras_augmentation.png` - imagens para o README
- `erros_FN.png`, `erros_FP.png`, `erros_TP.png`, `erros_TN.png` - análise qualitativa
- `historia_cnn1.json` e `historia_cnn2.json` - dados brutos das curvas

**Próximo notebook**: app de demonstração em Streamlit que carrega
`modelo_final_piro.keras` e classifica imagens novas.
